# 03 Model 1: Word Embeddings + Classifier
Training of architecture 1: frozen pretrained word embeddings with a 2-layer classifier (ReLU non-linearity). Only the classifier is trained. Experiments are tracked with Weights & Biases.

Load the features and labels that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only evaluated once at the end, in `05_evaluation.ipynb`.

In [23]:
import numpy as np
import torch

data = np.load("features/model1.npz")
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)

torch.Size([16801, 1200]) torch.float32
torch.Size([16801]) torch.int64


## Classifier
The 2-layer classifier like in the course exercise "PyTorch" (Week 1): linear layer, ReLU, linear layer. Input: 1200 features, output: 2 classes (h1 or h2).

In [24]:
import torch.nn as nn

class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
print(model)

Classifier(
  (linear1): Linear(in_features=1200, out_features=100, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=100, out_features=2, bias=True)
)


Number of parameters per layer (weights and biases), as in the course exercise "PyTorch" (Week 1).

In [25]:
for name, params in model.named_parameters():
    print(name, params.shape)

linear1.weight torch.Size([100, 1200])
linear1.bias torch.Size([100])
linear2.weight torch.Size([2, 100])
linear2.bias torch.Size([2])


Test the untrained classifier on the first 5 training examples: one pair of scores (h1, h2) per example.

In [26]:
x = X_train[:5]
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 1200])
torch.Size([5, 2])
tensor([[-0.1848, -0.1143],
        [-0.1771, -0.0636],
        [-0.1528, -0.0544],
        [-0.1209, -0.0621],
        [-0.1661, -0.0300]], grad_fn=<AddmmBackward0>)


## Loss
Cross-entropy loss for the 2 classes (course slides "Preliminaries", slide 27). `nn.CrossEntropyLoss` applies the softmax to the raw scores itself, so the classifier has no softmax layer. A binary decision can be made with 1 output and a sigmoid or with 2 outputs and a softmax (course slides "Project Discussions" (Week 5), slide 22). 2 outputs are used, one score per hypothesis, as in the classifier of the course exercise "PyTorch", so the prediction is the hypothesis with the higher score (`argmax`). A softmax layer in the model would apply the softmax twice (the "double sigmoid" warning on slide 30 of the same slides, Week 6). Before training, the loss should be about -log(0.5) ≈ 0.69, the value of random guessing between 2 classes.

In [27]:
criterion = nn.CrossEntropyLoss()

y = y_train[:5]
loss = criterion(output, y)
print(y)
print(loss)

tensor([1, 1, 0, 0, 1])
tensor(0.6781, grad_fn=<NllLossBackward0>)


## One training step 
The Adam optimizer updates the weights (course slides "Preliminaries", slide 29). Adam is used rather than AdamW (course slides "Project Discussions" (Week 6), slide 31): AdamW only changes how weight decay is applied, and no weight decay is used here, so both would make the same update steps (algorithm in the PyTorch documentation `torch.optim.AdamW`; Loshchilov & Hutter, 2019). One training step on the 5 examples as in the course exercise "PyTorch" (Week 1): forward pass, loss, backpropagation, update. Afterwards, the loss on the same 5 examples should be lower.

In [28]:
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=0.001)

output = model(x)
loss = criterion(output, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(loss)
print(criterion(model(x), y))

tensor(0.6781, grad_fn=<NllLossBackward0>)
tensor(0.6114, grad_fn=<NllLossBackward0>)


## Batches
The data is split into batches with a `DataLoader`, as in the course exercise "Experiment Tracking": training data in batches of 32, shuffled in every epoch. The course recommends the largest batch size that fits into the GPU memory (course slides "Project Discussions" (Week 4), slide 14); here the training runs on the CPU, where memory is not the limit, and one epoch takes only a few seconds, so the batch size of the course exercise is kept. 16801 = 525 x 32 + 1, so the last batch would contain a single example; `drop_last=True` skips it (a different example in every epoch, because of the shuffling). For measuring loss and accuracy, all batches have the same size, so that the average of the batch losses equals the average over all examples: validation data in 10 batches of 100, training data in 53 batches of 317 (16801 = 53 x 317), both not shuffled. `TensorDataset` pairs each row of features with its label.

In [29]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True, drop_last=True)
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

x_batch, y_batch = next(iter(train_loader))
print(len(train_loader), len(train_eval_loader), len(valid_loader))
print(x_batch.shape, y_batch.shape)

525 53 10
torch.Size([32, 1200]) torch.Size([32])


## Evaluation
Loss and accuracy on a dataset, batch by batch, as in the validation of the course exercise "Experiment Tracking": the loss of each batch and the predictions (`argmax`) and labels are collected; at the end, the loss is the average of the batch losses and the accuracy is the share of correct predictions over all examples. `model.eval()` and `torch.no_grad()` as in the `test_loop` of the PyTorch tutorial "Optimizing Model Parameters". A new untrained model is tested first: it should be at about 0.69 loss and 50% accuracy.

In [30]:
def evaluate(model, loader):
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            losses.append(criterion(output, y).item())
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100)
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6943, accuracy: 49.10%


## Experiment tracking
The training runs are tracked with Weights and Biases (W&B), as in the course exercise "Experiment Tracking". Log in to W&B (the API key was saved once with `wandb login` in the terminal).

In [31]:
import wandb
wandb.login()

True

## Training
Training loop as in the PyTorch tutorial "Optimizing Model Parameters": in every epoch, one training step (as in the course exercise "PyTorch") for each batch of the training data, then evaluation on the training and validation data. The training is written as a function `train()` with the hyperparameters as arguments, as in the course exercise "Experiment Tracking", so that it can be run with different hyperparameters. Every call trains a fresh model with seed 42 and creates one W&B run. Each run is logged to W&B: the hyperparameters in `config` and, after every epoch, the training and validation loss and accuracy with `wandb.log` (with `epoch` for the x-axis). `wandb.finish()` ends the run. After every epoch, the model is saved to `checkpoints/` if its validation loss is the lowest so far (`torch.save` of the `state_dict`, as in the PyTorch tutorial "Save and Load the Model"). At the end, the weights of the best epoch are loaded again, so `train()` returns the best model, not the one after the last epoch. The validation loss is used because it changes more smoothly than the accuracy. In addition, the best validation loss, the validation accuracy of that epoch and the best epoch are written to the run summary in W&B, so that the runs can be sorted by their best validation loss (by default, the summary shows the values of the last epoch). The cell ends with a first run with the default hyperparameters (10 epochs) as a walk-through; the hyperparameter experiments follow in the section "Experiments".

Further decisions: the learning rate stays constant (no learning rate schedule), the simplest choice, which can also work well (course slides "Project Discussions" (Week 6), slide 33, with Hägele et al., 2024). No early stopping: one epoch of model 1 takes only a few seconds, so all epochs are trained and the epoch with the lowest validation loss is kept afterwards; this gives the same model as early stopping with a long enough patience (same slides, slide 38), and the full curves in W&B show the overfitting. Model 2, where one epoch takes 1 to 2 minutes, uses early stopping. No dropout or weight decay (same slides, slide 34): selecting the best epoch by the validation loss already keeps the model from before it overfits (see the conclusion); dropout is tested for model 2 (`04_model2_rnn.ipynb`). No normalization layers in the chosen configuration (same slide): layer normalization is tested once in the section "Layer normalization" and does not lower the validation loss (see the conclusion).

In [10]:
import os

os.makedirs("checkpoints", exist_ok=True)

def train(inner_dim=100, learning_rate=0.001, num_epochs=10):
    name = f"glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
    checkpoint_path = f"checkpoints/{name}.pth"
    wandb.init(
        project="nlp-project1-art",
        name=name,
        config={
            "model": "glove-mlp",
            "inner_dim": inner_dim,
            "learning_rate": learning_rate,
            "epochs": num_epochs,
            "batch_size": train_loader.batch_size,
            "seed": 42,
        },
    )

    torch.manual_seed(42)
    model = Classifier(1200, 2, inner_dim=inner_dim)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return model

model = train(inner_dim=100, learning_rate=0.001)

Epoch 1: train loss 0.6693, train acc 60.63%, valid loss 0.6751, valid acc 58.70%
Epoch 2: train loss 0.6449, train acc 62.09%, valid loss 0.6654, valid acc 57.30%
Epoch 3: train loss 0.6251, train acc 64.97%, valid loss 0.6576, valid acc 60.50%
Epoch 4: train loss 0.6187, train acc 65.10%, valid loss 0.6590, valid acc 59.60%
Epoch 5: train loss 0.6051, train acc 66.71%, valid loss 0.6511, valid acc 61.50%
Epoch 6: train loss 0.6058, train acc 66.32%, valid loss 0.6654, valid acc 59.90%
Epoch 7: train loss 0.5844, train acc 68.74%, valid loss 0.6530, valid acc 62.20%
Epoch 8: train loss 0.5737, train acc 69.86%, valid loss 0.6494, valid acc 62.70%
Epoch 9: train loss 0.5630, train acc 70.32%, valid loss 0.6572, valid acc 60.20%
Epoch 10: train loss 0.5458, train acc 72.52%, valid loss 0.6500, valid acc 63.30%


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▂▄▄▅▄▆▆▇█
train_loss,█▇▅▅▄▄▃▃▂▁
val_acc,▃▁▅▄▆▄▇▇▄█
val_loss,█▅▃▄▁▅▂▁▃▁
best_epoch,8
best_val_acc,0.627
best_val_loss,0.64944
epoch,10
train_acc,0.72519
train_loss,0.54578


Best epoch: 8, valid loss 0.6494, valid acc 62.70%


Check that the returned model is the one of the best epoch: its validation loss and accuracy must be the same as in that epoch.

In [11]:
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6494, accuracy: 62.70%


## Experiments
Hyperparameter tuning on the validation set (course slides "Course Project HS26", slide 10: the validation set is used to compare model settings and to select the best model and hyperparameters). The experiments answer two questions: how large does the classifier have to be (hidden size), and which learning rate works, "the most important hyperparameter" (course slides "Project Discussions" (Week 6), slide 32)? The hidden size of the classifier (50, 100, 300) and the learning rate (0.001, 0.01) are varied in a grid search (all 6 combinations, course slides "Project Discussions" (Week 7), slide 44), each for 20 epochs. In every run, the best epoch is selected by the validation loss. Because the seed is set inside `train()`, the two runs with the same hidden size start from the same initial weights and see the batches in the same order, so the difference between them comes only from the learning rate. Each configuration is trained once. With seed 42, a run is exactly reproducible (the first 10 epochs of the 10-epoch run above and of the 20-epoch run with the same settings are identical), but the differences between configurations can partly be chance; the conclusion compares them with the standard error of a difference between two accuracies.

In [12]:
for inner_dim in [50, 100, 300]:
    for learning_rate in [0.001, 0.01]:
        train(inner_dim=inner_dim, learning_rate=learning_rate, num_epochs=20)

Epoch 1: train loss 0.6716, train acc 59.84%, valid loss 0.6774, valid acc 58.10%
Epoch 2: train loss 0.6465, train acc 63.16%, valid loss 0.6638, valid acc 58.20%
Epoch 3: train loss 0.6361, train acc 63.41%, valid loss 0.6631, valid acc 59.70%
Epoch 4: train loss 0.6242, train acc 65.13%, valid loss 0.6533, valid acc 61.00%
Epoch 5: train loss 0.6180, train acc 65.81%, valid loss 0.6509, valid acc 60.40%
Epoch 6: train loss 0.6158, train acc 65.93%, valid loss 0.6554, valid acc 60.10%
Epoch 7: train loss 0.6127, train acc 66.07%, valid loss 0.6527, valid acc 61.00%
Epoch 8: train loss 0.6175, train acc 65.13%, valid loss 0.6656, valid acc 60.40%
Epoch 9: train loss 0.6164, train acc 65.08%, valid loss 0.6538, valid acc 61.90%
Epoch 10: train loss 0.6096, train acc 66.24%, valid loss 0.6616, valid acc 60.10%
Epoch 11: train loss 0.6034, train acc 66.74%, valid loss 0.6574, valid acc 60.20%
Epoch 12: train loss 0.6001, train acc 66.73%, valid loss 0.6554, valid acc 62.20%
Epoch 13: tra

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▄▅▅▅▅▅▅▆▆▆▇▆▇▇▆▇█▇
train_loss,█▆▆▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁
val_acc,▁▁▄▆▅▄▆▅▇▄▅█▇▆▇▅▅▅▇▅
val_loss,█▄▄▂▁▂▂▅▂▄▃▂▃▄▁▃▄▇▂▄
best_epoch,15
best_val_acc,0.619
best_val_loss,0.65046
epoch,20
train_acc,0.68639
train_loss,0.57781


Best epoch: 15, valid loss 0.6505, valid acc 61.90%


Epoch 1: train loss 0.6941, train acc 50.36%, valid loss 0.6937, valid acc 50.80%
Epoch 2: train loss 0.6932, train acc 49.64%, valid loss 0.6933, valid acc 49.20%
Epoch 3: train loss 0.6931, train acc 50.36%, valid loss 0.6930, valid acc 50.80%
Epoch 4: train loss 0.6934, train acc 50.36%, valid loss 0.6931, valid acc 50.80%
Epoch 5: train loss 0.6937, train acc 50.36%, valid loss 0.6934, valid acc 50.80%
Epoch 6: train loss 0.6934, train acc 50.36%, valid loss 0.6932, valid acc 50.80%
Epoch 7: train loss 0.6931, train acc 49.66%, valid loss 0.6933, valid acc 49.20%
Epoch 8: train loss 0.6931, train acc 49.66%, valid loss 0.6932, valid acc 49.20%
Epoch 9: train loss 0.6931, train acc 49.66%, valid loss 0.6933, valid acc 49.20%
Epoch 10: train loss 0.6935, train acc 49.66%, valid loss 0.6939, valid acc 49.20%
Epoch 11: train loss 0.6931, train acc 49.66%, valid loss 0.6932, valid acc 49.20%
Epoch 12: train loss 0.6933, train acc 49.66%, valid loss 0.6935, valid acc 49.20%
Epoch 13: tra

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,█▁████▁▁▁▁▁▁▁█▁███▁█
train_loss,█▂▂▃▅▃▂▁▂▄▁▃▃▁▆▁▂▂▄▂
val_acc,█▁████▁▁▁▁▁▁▁█▁███▁█
val_loss,▅▃▁▂▃▂▃▂▃▆▂▄▄▁█▁▁▁▆▁
best_epoch,3
best_val_acc,0.508
best_val_loss,0.69302
epoch,20
train_acc,0.5036
train_loss,0.69311


Best epoch: 3, valid loss 0.6930, valid acc 50.80%


... (8 outputs left out)

Best epoch: 3, valid loss 0.6550, valid acc 60.20%


Epoch 1: train loss 0.6927, train acc 50.40%, valid loss 0.6928, valid acc 50.80%
Epoch 2: train loss 0.6733, train acc 56.44%, valid loss 0.6905, valid acc 55.10%
Epoch 3: train loss 0.6372, train acc 63.92%, valid loss 0.6645, valid acc 58.70%
Epoch 4: train loss 0.6226, train acc 65.50%, valid loss 0.6573, valid acc 61.10%
Epoch 5: train loss 0.6232, train acc 65.69%, valid loss 0.6575, valid acc 59.20%
Epoch 6: train loss 0.6277, train acc 64.40%, valid loss 0.6662, valid acc 60.20%
Epoch 7: train loss 0.6104, train acc 66.74%, valid loss 0.6540, valid acc 62.70%
Epoch 8: train loss 0.6143, train acc 66.05%, valid loss 0.6611, valid acc 62.00%
Epoch 9: train loss 0.6384, train acc 63.33%, valid loss 0.6907, valid acc 58.20%
Epoch 10: train loss 0.6051, train acc 67.13%, valid loss 0.6563, valid acc 62.40%
Epoch 11: train loss 0.6190, train acc 65.35%, valid loss 0.6716, valid acc 57.90%
Epoch 12: train loss 0.6013, train acc 67.56%, valid loss 0.6592, valid acc 62.00%
Epoch 13: tra

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▆▇▇▆▇▇▆█▇█▇█▇█████
train_loss,█▇▄▃▃▄▂▃▄▂▃▂▂▂▂▁▁▂▁▂
val_acc,▁▃▆▇▆▆█▇▅█▅▇▆█▇█▇██▇
val_loss,██▃▂▂▃▁▂█▁▄▂▂▄▆▄▄▄▇▂
best_epoch,7
best_val_acc,0.627
best_val_loss,0.65395
epoch,20
train_acc,0.68157
train_loss,0.59756


Best epoch: 7, valid loss 0.6540, valid acc 62.70%


## Learning rate 0.0001
The best learning rate of the grid, 0.001, is its smallest value, on the boundary of the tested range. Following the course slides "Project Discussions" (Week 7), slide 53 ("Don't stop testing hyperparameter values when the best performance has a value on the boundary of your testing range"), the next smaller value in exponential steps (slide 52), 0.0001, is tested too, for all three hidden sizes. With 10 times smaller update steps, the model needs more epochs, so these runs have 100 epochs instead of 20 (slide 41: select a number of epochs that makes sense).

In [13]:
# the next smaller learning rate, with more epochs (smaller steps)
for inner_dim in [50, 100, 300]:
    train(inner_dim=inner_dim, learning_rate=0.0001, num_epochs=100)

Epoch 1: train loss 0.6862, train acc 53.65%, valid loss 0.6877, valid acc 53.70%
Epoch 2: train loss 0.6741, train acc 61.29%, valid loss 0.6792, valid acc 58.70%
Epoch 3: train loss 0.6628, train acc 62.04%, valid loss 0.6719, valid acc 57.00%
Epoch 4: train loss 0.6541, train acc 62.52%, valid loss 0.6667, valid acc 57.30%
Epoch 5: train loss 0.6454, train acc 63.57%, valid loss 0.6620, valid acc 59.60%
Epoch 6: train loss 0.6387, train acc 64.05%, valid loss 0.6599, valid acc 60.40%
Epoch 7: train loss 0.6335, train acc 64.63%, valid loss 0.6564, valid acc 59.30%
Epoch 8: train loss 0.6286, train acc 65.02%, valid loss 0.6569, valid acc 61.10%
Epoch 9: train loss 0.6238, train acc 65.51%, valid loss 0.6545, valid acc 60.70%
Epoch 10: train loss 0.6206, train acc 65.89%, valid loss 0.6544, valid acc 60.00%
Epoch 11: train loss 0.6162, train acc 66.07%, valid loss 0.6533, valid acc 61.30%
Epoch 12: train loss 0.6128, train acc 66.54%, valid loss 0.6529, valid acc 61.00%
Epoch 13: tra

epoch,▁▁▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇█████
train_acc,▁▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
train_loss,█▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▁▁▂▁▁
val_acc,▁▅▄▆▇▆▆▅▇█▇▇███▇██▇▆▇█▇▇▇██▆▇▇▆▆▇▆▇▇▆▇▅▇
val_loss,▃▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▅▅▆▅▆▆▇█▇█
best_epoch,23
best_val_acc,0.621
best_val_loss,0.65046
epoch,100
train_acc,0.90852
train_loss,0.29182


Best epoch: 23, valid loss 0.6505, valid acc 62.10%


Epoch 1: train loss 0.6832, train acc 59.63%, valid loss 0.6850, valid acc 59.90%
Epoch 2: train loss 0.6715, train acc 58.30%, valid loss 0.6779, valid acc 55.20%
Epoch 3: train loss 0.6563, train acc 62.61%, valid loss 0.6675, valid acc 59.40%
Epoch 4: train loss 0.6467, train acc 63.03%, valid loss 0.6638, valid acc 59.30%
Epoch 5: train loss 0.6401, train acc 63.44%, valid loss 0.6598, valid acc 59.40%
Epoch 6: train loss 0.6336, train acc 63.97%, valid loss 0.6608, valid acc 58.60%
Epoch 7: train loss 0.6243, train acc 65.23%, valid loss 0.6560, valid acc 60.80%
Epoch 8: train loss 0.6198, train acc 65.77%, valid loss 0.6547, valid acc 61.40%
Epoch 9: train loss 0.6136, train acc 66.42%, valid loss 0.6512, valid acc 60.70%
Epoch 10: train loss 0.6122, train acc 66.44%, valid loss 0.6560, valid acc 61.00%
Epoch 11: train loss 0.6073, train acc 66.64%, valid loss 0.6550, valid acc 61.10%
Epoch 12: train loss 0.5998, train acc 67.71%, valid loss 0.6506, valid acc 62.50%
Epoch 13: tra

epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██
train_acc,▁▁▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▆▇▇▇▇▇▇████████
train_loss,██▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
val_acc,▁▁▁▄▃▄▅▅▄▅▅▅▆▆█▆▅█▅▃▆▅▅▅▇▇▅▇▄▅▆▄▆▄▅▇▅▄▄▆
val_loss,▂▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▂▂▂▂▃▃▃▃▃▄▃▃▄▄▅▅▅▆▆▆▇▇█
best_epoch,21
best_val_acc,0.626
best_val_loss,0.64892
epoch,100
train_acc,0.99
train_loss,0.11475


Best epoch: 21, valid loss 0.6489, valid acc 62.60%


Epoch 1: train loss 0.6767, train acc 60.97%, valid loss 0.6809, valid acc 58.20%
Epoch 2: train loss 0.6655, train acc 58.58%, valid loss 0.6749, valid acc 54.30%
Epoch 3: train loss 0.6577, train acc 59.14%, valid loss 0.6745, valid acc 56.00%
Epoch 4: train loss 0.6339, train acc 64.37%, valid loss 0.6597, valid acc 61.00%
Epoch 5: train loss 0.6243, train acc 65.18%, valid loss 0.6543, valid acc 60.00%
Epoch 6: train loss 0.6148, train acc 66.63%, valid loss 0.6541, valid acc 61.00%
Epoch 7: train loss 0.6054, train acc 67.48%, valid loss 0.6507, valid acc 61.90%
Epoch 8: train loss 0.5981, train acc 67.97%, valid loss 0.6541, valid acc 61.50%
Epoch 9: train loss 0.5923, train acc 68.13%, valid loss 0.6573, valid acc 61.00%
Epoch 10: train loss 0.5791, train acc 70.07%, valid loss 0.6516, valid acc 61.20%
Epoch 11: train loss 0.5665, train acc 71.44%, valid loss 0.6487, valid acc 62.20%
Epoch 12: train loss 0.5537, train acc 72.23%, valid loss 0.6500, valid acc 60.90%
Epoch 13: tra

epoch,▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▇▇▇▇▇▇▇▇█████
train_acc,▁▂▂▂▂▃▃▃▄▄▅▅▆▆▆▇▇███████████████████████
train_loss,██▇▇▇▆▆▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▅▇▅█▇▅▇▇▆█▇▆▇▆▇▆▆▆▆▅█▆▆█▆▆▆▇▆▆▆▇▆▇▆▆▆█▇
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▆▇▇▇█
best_epoch,14
best_val_acc,0.629
best_val_loss,0.64789
epoch,100
train_acc,1
train_loss,0.00089


Best epoch: 14, valid loss 0.6479, valid acc 62.90%


## Extending the tested range: learning rate 0.00001 and hidden size 1000
The best run so far, hidden size 300 with learning rate 0.0001, is on the boundary of the tested range twice: it has the smallest learning rate and the largest hidden size. Following slide 53 of the course slides "Project Discussions" (Week 7), the next value is tested in both directions, each with the other setting of the best run: learning rate 0.00001, the next exponential step (slide 52), with 300 epochs because the update steps are 10 times smaller, and hidden size 1000, about 3 times larger as from 100 to 300, with 100 epochs.

In [ ]:
# test the next values: a smaller learning rate and a larger hidden size
for inner_dim, learning_rate, num_epochs in [(300, 0.00001, 300), (1000, 0.0001, 100)]:
    train(inner_dim=inner_dim, learning_rate=learning_rate, num_epochs=num_epochs)

Epoch 1: train loss 0.6912, train acc 53.63%, valid loss 0.6918, valid acc 53.40%
Epoch 2: train loss 0.6893, train acc 56.96%, valid loss 0.6906, valid acc 54.80%
Epoch 3: train loss 0.6871, train acc 58.41%, valid loss 0.6888, valid acc 55.20%
Epoch 4: train loss 0.6849, train acc 58.49%, valid loss 0.6871, valid acc 55.50%
Epoch 5: train loss 0.6825, train acc 60.31%, valid loss 0.6853, valid acc 57.30%
Epoch 6: train loss 0.6801, train acc 60.82%, valid loss 0.6836, valid acc 56.60%
Epoch 7: train loss 0.6777, train acc 60.76%, valid loss 0.6821, valid acc 58.90%
Epoch 8: train loss 0.6757, train acc 59.63%, valid loss 0.6804, valid acc 55.80%
Epoch 9: train loss 0.6734, train acc 60.34%, valid loss 0.6796, valid acc 59.50%
Epoch 10: train loss 0.6704, train acc 61.67%, valid loss 0.6769, valid acc 59.60%
Epoch 11: train loss 0.6680, train acc 62.07%, valid loss 0.6753, valid acc 58.40%
Epoch 12: train loss 0.6658, train acc 62.18%, valid loss 0.6739, valid acc 58.40%
Epoch 13: tra

## Layer normalization
Does layer normalization lower the validation loss of the chosen configuration? The course lists normalization as an option (course slides "Project Discussions" (Week 6), slide 34) and layer norm yes/no as an example of an experiment (same slides (Week 7), slide 44). The chosen configuration (hidden size 300, learning rate 0.0001, 100 epochs) is trained once more with `nn.LayerNorm` between the first linear layer and the ReLU, because Ba et al. (2016) apply it before the nonlinearity. The check below shows that both runs start from the same weights and get the same first batch, so any difference comes from the layer norm; one run cannot show whether a small difference is more than chance. As for all runs, the best epoch is selected by the validation loss, and if this run has the lowest validation loss of all runs, it becomes the chosen configuration. The training steps are the same as in `train()`; only the class differs. Every 10th epoch is printed; all epochs are logged to W&B.

In [21]:
# the same classifier, with layer normalization between the first linear layer and the ReLU
class LayerNormClassifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        # new: normalizes the inner_dim values of each example
        self.norm = nn.LayerNorm(inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        # [batch, inner_dim]
        out = self.linear1(x)
        # new: same shape; mean 0, standard deviation 1 per example, then a learned scale and shift per unit
        out = self.norm(out)
        # negative values become 0
        out = self.relu(out)
        # [batch, 2]: one score per hypothesis
        out = self.linear2(out)
        return out


In [22]:
# same seed as in train(): the model with layer norm, then the first batch
torch.manual_seed(42)
model_ln = LayerNormClassifier(1200, 2, inner_dim=300)
x_ln, _ = next(iter(train_loader))

# the same for the model without layer norm
torch.manual_seed(42)
model_plain = Classifier(1200, 2, inner_dim=300)
x_plain, _ = next(iter(train_loader))

# same starting weights and same first batch?
print(torch.equal(model_ln.linear1.weight, model_plain.linear1.weight))
print(torch.equal(model_ln.linear2.weight, model_plain.linear2.weight))
print(torch.equal(x_ln, x_plain))
# how many parameters does layer norm add?
print(sum(p.numel() for p in model_ln.parameters()) - sum(p.numel() for p in model_plain.parameters()))


True
True
True
600


In [32]:
# the chosen configuration of model 1, with layer norm
inner_dim, learning_rate, num_epochs = 300, 0.0001, 100
name = f"glove-mlp-ln-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art",
    name=name,
    config={
        "model": "glove-mlp-ln",
        "inner_dim": inner_dim,
        "learning_rate": learning_rate,
        "epochs": num_epochs,
        "batch_size": train_loader.batch_size,
        "seed": 42,
    },
)

# same training as in train(), with the layer norm class
torch.manual_seed(42)
model = LayerNormClassifier(1200, 2, inner_dim=inner_dim)
optimizer = Adam(model.parameters(), lr=learning_rate)
best_valid_loss = float("inf")

for epoch in range(num_epochs):
    model.train()
    for x, y in train_loader:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_eval_loader)
    valid_loss, valid_acc = evaluate(model, valid_loader)
    # print every 10th epoch (all epochs are in W&B)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)

wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
print("For comparison, without layer norm (chosen run): best epoch 14, valid loss 0.6479, valid acc 62.90%")

Epoch 10: train loss 0.3303, train acc 88.42%, valid loss 0.7270, valid acc 60.00%
Epoch 20: train loss 0.0917, train acc 98.80%, valid loss 0.9448, valid acc 62.10%
Epoch 30: train loss 0.0311, train acc 99.90%, valid loss 1.3412, valid acc 61.70%
Epoch 40: train loss 0.0127, train acc 99.95%, valid loss 1.2943, valid acc 63.20%
Epoch 50: train loss 0.0059, train acc 100.00%, valid loss 1.4189, valid acc 62.90%
Epoch 60: train loss 0.0006, train acc 100.00%, valid loss 1.5307, valid acc 63.70%
Epoch 70: train loss 0.0003, train acc 100.00%, valid loss 1.6150, valid acc 63.50%
Epoch 80: train loss 0.0004, train acc 100.00%, valid loss 1.7507, valid acc 62.80%
Epoch 90: train loss 0.0001, train acc 100.00%, valid loss 1.7965, valid acc 63.10%
Epoch 100: train loss 0.0002, train acc 100.00%, valid loss 1.8948, valid acc 62.30%


epoch,▁▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇██
train_acc,▁▄▅▅▆███▇███████████████████████████████
train_loss,█▇▇▇▆▅▅▅▃▂▂▁▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▅▅▅▄▆▇▆▆▆▃▆▇▆▅▃▇▇█▇▇▆▆███▇▇▇▇▄▅▇▆▇▇▇▇▆▆
val_loss,▁▁▁▁▁▂▂▃▂▃▄▅▄▅▄▅▅▅▅▄▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██
best_epoch,3
best_val_acc,0.614
best_val_loss,0.65536
epoch,100
train_acc,1
train_loss,0.00023


Best epoch: 3, valid loss 0.6554, valid acc 61.40%
For comparison, without layer norm (chosen run): best epoch 14, valid loss 0.6479, valid acc 62.90%


## Results
Load the best checkpoint of every experiment, evaluate it on the validation set, and sort the results by the validation loss (lowest first).

In [34]:
results = []
# all runs: hidden size, learning rate and number of epochs
runs = [(inner_dim, learning_rate, num_epochs) for inner_dim in [50, 100, 300]
        for learning_rate, num_epochs in [(0.0001, 100), (0.001, 20), (0.01, 20)]]
runs += [(300, 0.00001, 300), (1000, 0.0001, 100)]

for inner_dim, learning_rate, num_epochs in runs:
    model = Classifier(1200, 2, inner_dim=inner_dim)
    model.load_state_dict(torch.load(f"checkpoints/glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}.pth", weights_only=True))
    valid_loss, valid_acc = evaluate(model, valid_loader)
    results.append((valid_loss, valid_acc, inner_dim, learning_rate))

# the run with layer norm: the same evaluation, with its own class
model = LayerNormClassifier(1200, 2, inner_dim=300)
model.load_state_dict(torch.load("checkpoints/glove-mlp-ln-h300-lr0.0001-e100.pth", weights_only=True))
valid_loss, valid_acc = evaluate(model, valid_loader)
results.append((valid_loss, valid_acc, "300 with layer norm", 0.0001))

for valid_loss, valid_acc, inner_dim, learning_rate in sorted(results):
    print(f"h{inner_dim}, lr {learning_rate}: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")

h300, lr 0.0001: valid loss 0.6479, valid acc 62.90%
h100, lr 0.0001: valid loss 0.6489, valid acc 62.60%
h100, lr 0.001: valid loss 0.6494, valid acc 62.70%
h1000, lr 0.0001: valid loss 0.6495, valid acc 61.90%
h300, lr 1e-05: valid loss 0.6500, valid acc 62.10%
h50, lr 0.0001: valid loss 0.6505, valid acc 62.10%
h50, lr 0.001: valid loss 0.6505, valid acc 61.90%
h100, lr 0.01: valid loss 0.6524, valid acc 62.00%
h300, lr 0.01: valid loss 0.6540, valid acc 62.70%
h300, lr 0.001: valid loss 0.6550, valid acc 60.20%
h300 with layer norm, lr 0.0001: valid loss 0.6554, valid acc 61.40%
h50, lr 0.01: valid loss 0.6930, valid acc 50.80%


## Active ReLU units
With learning rate 0.01 and hidden size 50, the model predicts the same class for every example. Count how many hidden units are active, i.e. output a value greater than 0 after the ReLU for more than 1% of the training examples (169 or more of the 16801): before training, and in the best checkpoints of the nine runs with hidden sizes 50, 100, 300 and learning rates 0.01, 0.001, 0.0001. The 1% is a practical threshold, not the definition of a dead unit: a unit that outputs 0 for every example gets no gradient and no longer learns ("dying ReLU", Lu et al., 2020), while a rarely active unit can still learn. The untrained models are created with seed 42, so they have the same start weights as the training runs with the same hidden size.

In [33]:
# no training here, so no gradients are needed
with torch.no_grad():
    for inner_dim in [50, 100, 300]:
        # same start weights as the training runs with this hidden size
        torch.manual_seed(42)
        untrained = Classifier(1200, 2, inner_dim=inner_dim)
        # hidden layer only (linear1 + ReLU): one row per example, one column per unit
        hidden = untrained.relu(untrained.linear1(X_train))
        # share of the examples on which each unit outputs more than 0; active if more than 1%
        active = (hidden > 0).float().mean(dim=0) > 0.01
        # count the active units (True counts as 1)
        print(f"h{inner_dim}, untrained: {active.sum().item()} of {inner_dim} units active")
        # the same for the trained models
        for learning_rate, num_epochs in [(0.0001, 100), (0.001, 20), (0.01, 20)]:
            # load the weights of the best epoch of the run (-e20 / -e100 = the run had 20 / 100 epochs)
            model = Classifier(1200, 2, inner_dim=inner_dim)
            model.load_state_dict(torch.load(f"checkpoints/glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}.pth", weights_only=True))
            hidden = model.relu(model.linear1(X_train))
            active = (hidden > 0).float().mean(dim=0) > 0.01
            print(f"h{inner_dim}, lr {learning_rate}: {active.sum().item()} of {inner_dim} units active")

h50, untrained: 49 of 50 units active
h50, lr 0.0001: 50 of 50 units active
h50, lr 0.001: 5 of 50 units active
h50, lr 0.01: 0 of 50 units active
h100, untrained: 94 of 100 units active
h100, lr 0.0001: 98 of 100 units active
h100, lr 0.001: 17 of 100 units active
h100, lr 0.01: 1 of 100 units active
h300, untrained: 279 of 300 units active
h300, lr 0.0001: 293 of 300 units active
h300, lr 0.001: 40 of 300 units active
h300, lr 0.01: 1 of 300 units active


## Hypothesis-only check
How much of the result comes from the hypotheses alone? The chosen configuration (hidden size 300, learning rate 0.0001, 100 epochs) is trained once more with only h1 and h2 as input (the last 600 of the 1200 numbers), without O1 and O2. If it reaches about the same validation loss and accuracy, the observations add little, and most of the result can be reached by judging which hypothesis is more plausible on its own: a hypothesis-only baseline (Gururangan et al., 2018; Poliak et al., 2018; Bhagavatula et al., 2020, Table 1: BERT with the hypothesis only, 59.5% on the ART test set). This run is a diagnostic baseline, not a candidate for model 1: the task is to choose the hypothesis that explains the observations, so the input of model 1 stays all four texts. The training steps are the same as in `train()`; only the input size (600) and the loaders differ. Every 10th epoch is printed; all epochs are logged to W&B.

In [11]:
# only h1 and h2: the last 600 of the 1200 numbers (O1, O2, h1, h2, 300 each)
X_train_hyp = X_train[:, 600:]
X_valid_hyp = X_valid[:, 600:]
# the same batches as for the other runs
train_loader_hyp = DataLoader(TensorDataset(X_train_hyp, y_train), batch_size=32, shuffle=True, drop_last=True)
train_eval_loader_hyp = DataLoader(TensorDataset(X_train_hyp, y_train), batch_size=317)
valid_loader_hyp = DataLoader(TensorDataset(X_valid_hyp, y_valid), batch_size=100)

# the chosen configuration of model 1
inner_dim, learning_rate, num_epochs = 300, 0.0001, 100
name = f"glove-mlp-hyp-only-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art",
    name=name,
    config={
        "model": "glove-mlp-hyp-only",
        "inner_dim": inner_dim,
        "learning_rate": learning_rate,
        "epochs": num_epochs,
        "batch_size": train_loader_hyp.batch_size,
        "seed": 42,
    },
)

# same training as in train(), with 600 inputs instead of 1200
torch.manual_seed(42)
model = Classifier(600, 2, inner_dim=inner_dim)
optimizer = Adam(model.parameters(), lr=learning_rate)
best_valid_loss = float("inf")

for epoch in range(num_epochs):
    model.train()
    for x, y in train_loader_hyp:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_eval_loader_hyp)
    valid_loss, valid_acc = evaluate(model, valid_loader_hyp)
    # print every 10th epoch (all epochs are in W&B)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)

wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
print("For comparison, all four texts (chosen run): valid loss 0.6479, valid acc 62.90%")

Epoch 10: train loss 0.6031, train acc 67.40%, valid loss 0.6400, valid acc 64.00%
Epoch 20: train loss 0.5543, train acc 71.79%, valid loss 0.6405, valid acc 62.60%
Epoch 30: train loss 0.4943, train acc 77.27%, valid loss 0.6462, valid acc 63.70%
Epoch 40: train loss 0.4286, train acc 82.71%, valid loss 0.6497, valid acc 64.10%
Epoch 50: train loss 0.3699, train acc 86.60%, valid loss 0.6663, valid acc 63.60%
Epoch 60: train loss 0.3096, train acc 90.38%, valid loss 0.7023, valid acc 62.90%
Epoch 70: train loss 0.2591, train acc 93.08%, valid loss 0.7354, valid acc 63.20%
Epoch 80: train loss 0.2078, train acc 95.57%, valid loss 0.7726, valid acc 63.70%
Epoch 90: train loss 0.1653, train acc 97.29%, valid loss 0.8216, valid acc 62.00%
Epoch 100: train loss 0.1305, train acc 98.39%, valid loss 0.8878, valid acc 61.50%


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
train_acc,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█████
train_loss,████▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
val_acc,▁▂▃▃▄▅▆▇▇▇▅▆█▅▆▆▇▆▄▄▆▆▅▅▄▄▅▅▅▄▅▅▄▆▅▃▄▄▄▅
val_loss,▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▇▇██
best_epoch,28
best_val_acc,0.653
best_val_loss,0.63441
epoch,100
train_acc,0.98387
train_loss,0.13053


Best epoch: 28, valid loss 0.6344, valid acc 65.30%
For comparison, all four texts (chosen run): valid loss 0.6479, valid acc 62.90%


## Conclusion
- **Chosen configuration:** hidden size 300, learning rate 0.0001 (best epoch 14): validation loss 0.6479, accuracy 62.90%, the lowest validation loss of all 12 runs. Its checkpoint `checkpoints/glove-mlp-h300-lr0.0001-e100.pth` is evaluated once on the test set in `05_evaluation.ipynb`.
- **The hyperparameters barely matter:** all runs that learned are between 0.6479 and 0.6554 validation loss and between 60.20% and 62.90% accuracy. The standard error of the difference between two accuracies of about 62% on 1000 examples is about √(2 × 0.62 × 0.38 / 1000) ≈ 2.2 percentage points. All runs use the same examples, so this is only a rough guide, but the largest difference, 2.7 points, is about 1.2 of these standard errors, so the runs cannot be clearly separated. The limit is the input, not the classifier: the average word vectors hide the few words in which h1 and h2 differ (see `01_data_exploration.ipynb`).
- **Learning rate:** 0.01 is too high: with hidden size 50, no hidden unit is active on more than 1% of the training examples (0 of 50), so the model predicts h1 for every example (50.80%); with hidden sizes 100 and 300, only 1 unit is. 0.0001 is the best value and lies inside the tested range: 0.00001 is slightly worse with hidden size 300 (0.6500 vs. 0.6479, best epoch 69). Between 0.001 and 0.0001 the validation loss changes by at most 0.007 (hidden size 300: 0.6550 → 0.6479; 100: 0.6494 → 0.6489; 50: 0.6505 → 0.6505), about as much as between neighbouring epochs of one run (0.6494, 0.6572, 0.6500 in epochs 8 to 10 with hidden size 100). The loss is flat in this range.
- **Hidden size:** 300 lies inside the tested range: 1000 is slightly worse with learning rate 0.0001 (0.6495 vs. 0.6479, best epoch 6 instead of 14), and it reaches 100% training accuracy (validation loss 1.8758 after 100 epochs).
- **Layer normalization:** it fits the training data in fewer epochs but does not improve the best validation loss in this configuration. With the same starting weights and the same first batch, the best epoch is 3 instead of 14, and the training accuracy after 10 epochs is 88.42% instead of 70.07%. At their respective best epochs, the validation loss is 0.6554 with layer normalization versus 0.6479 without it, and the validation accuracy is 61.40% versus 62.90%. The configuration without it remains chosen. This comparison uses one learning rate and one seed, so it does not show whether layer normalization would help with other settings.
- **Few units are enough:** with learning rate 0.001, only 5 of 50, 17 of 100 and 40 of 300 units are active on more than 1% of the examples, and a model with a single such unit (hidden size 100 or 300, learning rate 0.01) still reaches 62.00% and 62.70%. With learning rate 0.0001, almost all units stay active (50, 98 and 293), with about the same validation loss. The averaged input provides very few useful features.
- **Hypothesis-only check:** with only h1 and h2 as input, the same configuration reaches a lower validation loss (0.6344 vs. 0.6479) and a higher accuracy (65.30% vs. 62.90%). The observations therefore give model 1 no measurable benefit on the validation set: a separately trained model does better with the hypotheses alone, and the averaged observations apparently only add noise. This does not show that model 1 ignores the observations, only that they do not help it. The hypothesis-only result works on the validation set, which comes from the unfiltered original training data (see "Data" in `README.md`); the test set was adversarially filtered against such artifacts (Bhagavatula et al., 2020), so model 1 is expected to lose much of its validation accuracy on the test set.
- **Larger hidden sizes overfit faster:** with hidden size 300, the training accuracy reaches 94.71% after 20 epochs with learning rate 0.001 (validation loss 0.6550 at the best epoch 3, 0.9564 at epoch 20) and 100% after 100 epochs with 0.0001 (validation loss 1.5785). Selecting the best epoch by the validation loss keeps the model from before the overfitting.

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier`, based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Parameters and test forward pass: `named_parameters()` and a forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss: cross-entropy from the course slides "Preliminaries" (Week 1), slide 27; `criterion` and `loss = criterion(output, target)` as in the course exercise "PyTorch" (Week 1); `nn.CrossEntropyLoss` (applies the softmax to the raw scores) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html.
- Random seed: fixed seed 42 as in the course exercise "Experiment Tracking" (`seed_everything(42)`); `torch.manual_seed` from the PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html.
- Optimizer and training step: Adam from the course slides "Preliminaries" (Week 1), slide 29; `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` as in the course exercise "PyTorch" (Week 1).
- Batches: `DataLoader` with `batch_size=32` and `shuffle=True` for training and no shuffling for validation as in the course exercise "Experiment Tracking"; `TensorDataset` and `drop_last=True` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html.
- Evaluation: collecting the batch losses, `argmax` predictions and labels, joining them with `torch.cat` and averaging the batch losses as in the validation of the course exercise "Experiment Tracking" (`validation_step` and `on_validation_epoch_end`); `model.eval()` and `torch.no_grad()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters", https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html; `np.mean` from NumPy (Harris et al., 2020).
- Training loop: epochs with a train loop over all batches and an evaluation after each epoch, and `model.train()`, as in the PyTorch tutorial "Optimizing Model Parameters"; the training step inside as in the course exercise "PyTorch" (Week 1); training as a function `train()` with the hyperparameters as arguments (with default values) and the seed set inside, as in the course exercise "Experiment Tracking".
- Experiment tracking: `wandb.login()`, `wandb.init(project=..., config=...)` and `wandb.finish()` as in the course exercise "Experiment Tracking"; run name `name=` from the W&B `wandb.init()` reference, https://docs.wandb.ai/models/ref/python/functions/init; logging the metrics once per epoch with `epoch` as x-axis as in the W&B documentation "Log objects and media", https://docs.wandb.ai/models/track/log (written there as `run.log`, which is the same as `wandb.log` for the current run); `train_loader.batch_size` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html; Weights and Biases (Biewald, 2020); `nbformat` installed for W&B, as in the course exercise "Experiment Tracking".
- Saving the best model: `torch.save(model.state_dict(), ...)` and `model.load_state_dict(torch.load(..., weights_only=True))` as in the PyTorch tutorial "Save and Load the Model", https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html; `os.makedirs` from the Python standard library, https://docs.python.org/3/library/os.html; `float("inf")` from the Python documentation, https://docs.python.org/3/library/functions.html#float; best values in the run summary (`run.summary[...]`, here `wandb.run.summary` for the current run) as in the W&B documentation "Log summary metrics", https://docs.wandb.ai/models/track/log/log-summary.
- Experiments: hyperparameter tuning on the validation set, as on slide 10 of the course slides "Course Project HS26" (compare model settings, select the best model and hyperparameters).
- Results: loading the checkpoints with `load_state_dict` as in the PyTorch tutorial "Save and Load the Model"; sorting the results with `sorted()` (tuples are compared by their first item first) as in the Python documentation "Sorting Techniques", https://docs.python.org/3/howto/sorting.html.
- Active ReLU units: term and explanation "dying ReLU" from Lu, Shin, Su & Karniadakis (2020), *Dying ReLU and Initialization: Theory and Numerical Examples*, Communications in Computational Physics, https://doi.org/10.4208/cicp.OA-2020-0165; `torch.no_grad()` as in the PyTorch tutorial "Optimizing Model Parameters"; share of examples per unit with `.float().mean(dim=0)` as in the evaluation (`torch.mean`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.mean.html); counting the active units with `.sum().item()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters".
- Conclusion: standard error of an accuracy √(p(1 − p)/n), from the normal approximation interval in "Binomial proportion confidence interval", Wikipedia, https://en.wikipedia.org/wiki/Binomial_proportion_confidence_interval; standard error of a difference between two accuracies √(p₁(1 − p₁)/n + p₂(1 − p₂)/n), from "Two-proportion Z-test", Wikipedia, https://en.wikipedia.org/wiki/Two-proportion_Z-test.
- Training decisions: course slides "Project Discussions": 1 output with a sigmoid or 2 outputs with a softmax (Week 5, slide 22), "double sigmoid" (Week 6, slide 30), Adam and AdamW (Week 6, slide 31), the learning rate as the most important hyperparameter (Week 6, slide 32), a constant learning rate (Week 6, slide 33, citing Hägele, A., Bakouch, E., Kosson, A., Ben Allal, L., Von Werra, L., & Jaggi, M. (2024), *Scaling Laws and Compute-Optimal Training Beyond Fixed Training Durations*, https://arxiv.org/abs/2405.18392), regularization (Week 6, slide 34), early stopping (Week 6, slide 38), batch size (Week 4, slide 14). AdamW: algorithm in the PyTorch documentation `torch.optim.AdamW`, https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html, from Loshchilov & Hutter (2019), *Decoupled Weight Decay Regularization*, ICLR 2019, https://arxiv.org/abs/1711.05101.
- Grid search, learning rate 0.0001, learning rate 0.00001 and hidden size 1000: grid search (all combinations) from the course slides "Project Discussions" (Week 7), slide 44; testing beyond the boundary of the tested range (slide 53), exponentially changing values (slide 52) and a sensible number of epochs (slide 41) from the same slides.
- Hypothesis-only check: hypothesis-only baselines from Gururangan, S., Swayamdipta, S., Levy, O., Schwartz, R., Bowman, S., & Smith, N. A. (2018), *Annotation Artifacts in Natural Language Inference Data*, NAACL 2018, https://arxiv.org/abs/1803.02324, and Poliak, A., Naradowsky, J., Haldar, A., Rudinger, R., & Van Durme, B. (2018), *Hypothesis Only Baselines in Natural Language Inference*, *SEM 2018, https://arxiv.org/abs/1805.01042; for ART: Bhagavatula et al. (2020), Table 1 (BERT-ft [h_i Only], 59.5%); column indexing `[:, ...]` (here `X_train[:, 600:]`) as in the course exercise "PyTorch" (Week 1); training steps as in `train()` above; origin of the splits: Bhagavatula et al. (2020), Table 6, and course slides "Course Project HS26", slide 8.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md; all code reviewed and understood by me.
- Layer normalization: `nn.LayerNorm` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html; placed between the linear layer and the ReLU following Ba, Kiros & Hinton (2016), *Layer Normalization* ("after the normalization but before the non-linearity"), https://arxiv.org/abs/1607.06450; normalization as an option and layer norm (yes/no) as an experiment from the course slides "Project Discussions" (Week 6), slide 34, and (Week 7), slide 44. Check: `torch.equal`, https://docs.pytorch.org/docs/stable/generated/torch.equal.html, and `torch.numel`, https://docs.pytorch.org/docs/stable/generated/torch.numel.html, from the PyTorch documentation.